1. Business Questions for the Columns

is_valid_purchase: "Did this order result in an actual sale, rather than being canceled or unavailable?"
is_approved: "Did the customer's payment successfully go through?"
is_shipped / is_delivered: "Did the product actually leave the warehouse and reach the customer?"
is_on_time: "Are our logistics partners keeping their delivery timeline promises to the customer?"
promised_days: "How long are we telling customers they have to wait for their items?"
item_value / freight_value: "How much revenue did the merchandise generate, and how much did it cost to ship it?"
is_good_order: "Was the customer experience flawless (on time and highly rated)?"
is_first_purchase: "Is this a new customer we just acquired, or a returning customer?"


2. Decisions vs Facts Answers

1. GMV = SUM(price) (excluding freight)

Alternative rejected: Using payment_value (which was used in Project 1).
Why: payment_value includes shipping costs (freight), and potentially discounts or interest. We want GMV to reflect the strict value of the goods sold, not the total transaction cost including logistics.
2. A "good order" = delivered on time AND reviewed 4–5★

Alternative rejected: Assuming no review means "everything was fine" and counting it as a good order.
Why: We are choosing a very strict, conservative definition of quality. If an order was never reviewed, we don't know for sure if the customer was happy. We are sacrificing quantity (it will look like we have fewer good orders) to guarantee that every "good order" was explicitly confirmed by the customer.
3. A "valid purchase" excludes canceled and unavailable

Alternative rejected: Including all orders regardless of final status.
Why: We are excluding them because we are calculating final realized sales. A canceled order doesn't generate real revenue. However, if we were analyzing the sales funnel (how many people tried to checkout) or retention (did a user interact with us again), we would count canceled orders, because a cancellation is still a strong signal of purchase intent.


In [1]:
import sys
sys.path.append("..")

from src.metrics import connect_raw, build_order_facts, MARTS, START, END

con = connect_raw()
facts = build_order_facts(con)
print(f"{len(facts):,} orders, {facts['customer_unique_id'].nunique():,} customers")
print(facts.dtypes)

99,092 orders, 95,774 customers
order_id                          str
customer_unique_id                str
customer_state                    str
order_status                      str
purchase_date          datetime64[us]
purchase_month         datetime64[us]
is_valid_purchase                bool
is_approved                      bool
is_shipped                       bool
is_delivered                     bool
is_on_time                       bool
promised_days                   int64
item_value                    float64
freight_value                 float64
n_items                         Int64
n_sellers                       Int64
main_category                     str
review_score                    Int64
is_good_order                    bool
first_purchase_date    datetime64[us]
is_first_purchase                bool
dtype: object


In [5]:
assert facts["order_id"].is_unique, "join fan-out: an order appears twice"

assert (facts["is_good_order"] <= facts["is_on_time"]).all()
assert (facts["is_on_time"] <= facts["is_delivered"]).all()




In [3]:
direct = con.execute(f"""
    SELECT SUM(oi.price)
    FROM order_items oi JOIN orders o ON oi.order_id = o.order_id
    WHERE CAST(o.order_purchase_timestamp AS DATE) BETWEEN DATE '{START}' AND DATE '{END}'
""").fetchone()[0]
print(f"facts GMV {facts['item_value'].sum():,.0f} vs direct {direct:,.0f}")
assert abs(facts["item_value"].sum() - direct) < 1

facts GMV 13,541,713 vs direct 13,541,713


In [4]:
print((facts["main_category"] == "unknown").mean(), "share unknown")
print(facts["main_category"].value_counts().head(10))

0.02168691720825092 share unknown
main_category
bed_bath_table           9325
health_beauty            8757
sports_leisure           7664
computers_accessories    6652
furniture_decor          6279
housewares               5812
watches_gifts            5602
telephony                4169
auto                     3869
toys                     3845
Name: count, dtype: int64
